### 📘 1. Foundry Agent



In [19]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from dotenv import load_dotenv
load_dotenv()

from utils.config import get_env, get_credential

# Endpoints
PROJECT_ENDPOINT = get_env("AZURE_AI_PROJECT_ENDPOINT_FULL")  # for AIProjectClient
MODEL                 = get_env("AZURE_AI_MODEL_DEPLOYMENT_NAME")

AGENT_NAME = "ContosoConcierge-af"

print(f"📡 Project endpoint : {PROJECT_ENDPOINT}")
print(f"🧠 Model:                   {MODEL}")

📡 Project endpoint : https://foundry-demo-us-east-resource.services.ai.azure.com/api/projects/prj-foundry-demo-us-east-dev-001
🧠 Model:                   gpt-4o


---
### Part A — Create Agent with Foundry Project SDK 🏗️


In [11]:
# ── A1. Create the agent in Foundry ──────────────────────────────────────────
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

INSTRUCTIONS = """\
You are **Contoso Concierge**, a friendly and knowledgeable assistant for
Contoso Corporation. You help employees with general questions about company
policies, product information, and day-to-day tasks.

Guidelines:
- Be concise but thorough.
- When you don't know something, say so honestly.
- Use a professional yet warm tone.
"""

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=get_credential(),
)

agent_version = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=INSTRUCTIONS,
    ),
)

print(f"✅ Agent created in Foundry!")
print(f"   Name:    {agent_version.name}")
print(f"   Version: {agent_version.version}")
print(f"   ID:      {agent_version.id}")
print(f"\n🔍 Traces for this agent will appear in the Foundry portal.")

✅ Agent created in Foundry!
   Name:    ContosoConcierge-af
   Version: 1
   ID:      ContosoConcierge-af:1

🔍 Traces for this agent will appear in the Foundry portal.


---
### Part B — Use with Microsoft Agent Framework 🤖

In [12]:
# ── B1. Connect to the Foundry agent via Agent Framework ─────────────────────
from agent_framework import Agent
from agent_framework.foundry import FoundryAgent

af_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=agent_version.name,
    agent_version=str(agent_version.version),
    credential=get_credential(),
)

# ── B2. Single-turn call via Agent Framework ─────────────────────────────────
result = await af_agent.run("What departments does Contoso have?")
print(f"🗨️  Agent (via Agent Framework): {result}")

🗨️  Agent (via Agent Framework): Contoso Corporation has several key departments, including:

1. **Human Resources (HR)**: Manages employee relations, benefits, and recruitment.
2. **Finance and Accounting**: Handles budgeting, financial reporting, and payroll.
3. **Sales and Marketing**: Focuses on promoting and selling products and services.
4. **Research and Development (R&D)**: Responsible for product innovation and development.
5. **Information Technology (IT)**: Provides tech support and manages IT infrastructure.
6. **Customer Service**: Assists customers with inquiries and support.
7. **Operations**: Ensures efficient business processes and logistics.

If you need more specific details about any department, feel free to ask!


In [13]:
# ── B3. Streaming response ───────────────────────────────────────────────────
print("🗨️  Agent (streaming): ", end="", flush=True)
async for chunk in af_agent.run("Tell me how to make a soup.", stream=True):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print()

🗨️  Agent (streaming): Here’s a simple recipe for a basic vegetable soup:

**Ingredients:**
- 1 tablespoon olive oil
- 1 onion, chopped
- 2 cloves garlic, minced
- 3 carrots, sliced
- 2 celery stalks, chopped
- 1 bell pepper, chopped
- 1 zucchini, sliced
- 1 can (14.5 oz) diced tomatoes
- 6 cups vegetable broth
- 1 teaspoon dried thyme
- 1 teaspoon dried basil
- Salt and pepper to taste
- Optional: 1 cup cooked pasta or rice

**Instructions:**
1. **Heat the Oil:** In a large pot, heat the olive oil over medium heat.
2. **Sauté Vegetables:** Add the onion and garlic, sauté until the onion is translucent.
3. **Add More Veggies:** Add carrots, celery, bell pepper, and zucchini. Cook for about 5 minutes.
4. **Mix in Tomatoes and Broth:** Stir in the diced tomatoes and vegetable broth.
5. **Season:** Add thyme, basil, salt, and pepper.
6. **Simmer:** Bring to a boil, then reduce heat and let it simmer for about 20-25 minutes, or until the vegetables are tender.
7. **Optional Additions:** St

---
### Part C — Sessions and Conversations

In [14]:
# ── C1. Multi-turn with AgentSession ────────────────────────────
session = af_agent.create_session()
print(f"💬 Session created: {session.session_id}")

# First turn
result1 = await af_agent.run(
    "My name is Dibyendu and I work in IT department. I like chocolates.",
    session=session,
)
print(f"\n🗨️  Agent: {result1}")

# Second turn — agent should remember the name
result2 = await af_agent.run(
    "What do I like ?",
    session=session,
)
print(f"\n🗨️  Agent: {result2}")

💬 Session created: 815232be-1c5b-4274-bca7-71e791482d61

🗨️  Agent: Hi Dibyendu! It's great to meet you. Being in the IT department must be exciting. As for chocolates, they're always a nice treat! If there's anything specific you need assistance with, feel free to ask.

🗨️  Agent: You mentioned that you like chocolates! If there's anything else, feel free to share.


In [15]:
# ── C2. Multi-turn conversation with AgentSession and Foundry Conversation ────────────────────────────

# Create a new conversation in Foundry and link it to the AgentSession
conversation = project.get_openai_client().conversations.create()
print(f"\n🗨️  Conversation: {conversation.id}")



🗨️  Conversation: conv_09ab443d7383a38700sOlhRXTFe5Ki5sq3Szmn0FIUF9Q4buZe


In [16]:
# ── C3. Multi-turn with AgentSession and Foundry Conversation ────────────────────────────
session = af_agent.create_session()
session.service_session_id = conversation.id

print(f"💬 Session created: {session.session_id}")
print(f"🔗 Bound to Foundry conversation: {session.service_session_id}")

# First turn
result1 = await af_agent.run(
    "My name is Dibyendu and I work in IT department. I like chocolates.",
    session=session,
)
print(f"\n🗨️  Agent: {result1}")

# Second turn — agent should remember the name
result2 = await af_agent.run(
    "What do I like ?",
    session=session,
)
print(f"\n🗨️  Agent: {result2}")

💬 Session created: 8004c00d-531f-4b01-aa71-534931e87fcb
🔗 Bound to Foundry conversation: conv_09ab443d7383a38700sOlhRXTFe5Ki5sq3Szmn0FIUF9Q4buZe

🗨️  Agent: Nice to meet you, Dibyendu! Chocolates are a great choice. Is there something specific you'd like to know or discuss today?

🗨️  Agent: You mentioned that you like chocolates! Is there anything else you're interested in exploring or discussing?


In [17]:
# ── C4. Multi-turn with AgentSession and Existing Foundry Conversation ────────────────────────────
session = af_agent.create_session()
session.service_session_id = conversation.id
print(f"💬 Session created: {session.session_id}")
print(f"🔗 Reusing Foundry conversation: {session.service_session_id}")

# First turn
result1 = await af_agent.run(
    "What is my name ?",
    session=session,
)
print(f"\n🗨️  Agent: {result1}")

# Second turn — agent should remember the name
result2 = await af_agent.run(
    "What do I like ?",
    session=session,
)
print(f"\n🗨️  Agent: {result2}")

💬 Session created: 692f8b45-b907-4b9d-a578-fc24ae35ba55
🔗 Reusing Foundry conversation: conv_09ab443d7383a38700sOlhRXTFe5Ki5sq3Szmn0FIUF9Q4buZe

🗨️  Agent: Your name is Dibyendu. How can I assist you further today?

🗨️  Agent: You like chocolates! Anything else you'd like to add or ask about?


---
### Part C — Cleanup 🧹

In [18]:
# ── C1. Delete the agent version from Foundry ────────────────────────────────
# Uncomment the next line to delete the agent when you're done:
project.agents.delete(agent_name=AGENT_NAME)
print("🧹 Agent deleted from Foundry")

#print("💡 Tip: Keep the agent to reuse in other notebooks or the Foundry Playground.")
print(f"   Agent name:    {AGENT_NAME}")
print(f"   Agent version: {agent_version.version}")

🧹 Agent deleted from Foundry
   Agent name:    ContosoConcierge-af
   Agent version: 1


---
### ✅ What you demonstrated

| Capability | SDK / Feature |
|---|---|
| Register agent in Foundry | `AIProjectClient` → `agents.create_version()` |
| Server-side traces | All calls visible in Foundry portal → Tracing |
| Agent Framework bridge | `FoundryAgent(agent_name=..., agent_version=...)` |
| Multi-turn memory | `AgentSession` retains context across turns |
| Streaming | `async for chunk in agent.run(..., stream=True)` |

**Next →** [02_agent_tools_functioncall.ipynb](./02_agent_tools_functioncall.ipynb) — Local Function Tools.